<a href="https://colab.research.google.com/github/utpalssg/pythonai/blob/newBranch/TensorTry.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from getpass import getpass
token = getpass('Enter your GitHub token:')
!git clone --recurse-submodules https://{token}@github.com/utpalssg/pythonai.git

Enter your GitHub token:··········
Cloning into 'pythonai'...
remote: Enumerating objects: 87, done.
remote: Counting objects: 100% (87/87), done.
remote: Compressing objects: 100% (81/81), done.
remote: Total 87 (delta 41), reused 4 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (87/87), 1.85 MiB | 4.71 MiB/s, done.
Resolving deltas: 100% (41/41), done.


In [2]:
import torch
from torchvision import models, transforms

# using cpu
#model = models.resnet50(pretrained=True)

# using gpu
model = models.resnet50(pretrained=True).to("cuda")

/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth
100%|██████████| 97.8M/97.8M [00:00<00:00, 149MB/s]


In [3]:
from PIL import Image

img = Image.open("/content/pythonai/inference/img1.jpg")


In [4]:
from torchvision import transforms

transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])
# Apply the transform to the original PIL image from cell cOmxknVSJv8t
img_tensor = transform(img)
img_tensor.shape

torch.Size([3, 224, 224])

In [5]:
import torch

img_batch = torch.unsqueeze(img_tensor, 0).to("cuda")
img_batch.shape

torch.Size([1, 3, 224, 224])

In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = model.to(device)             # Move model to GPU
img_batch = img_batch.to(device)     # Move input to GPU

model.eval()
with torch.no_grad():
    outputs = model(img_batch)
probs = torch.nn.functional.softmax(outputs[0], dim=0)

In [7]:
import pandas as pd

labels = pd.read_csv("https://raw.githubusercontent.com/pytorch/hub/master/imagenet_classes.txt", header=None)
labels[0][3]

'tiger shark'

In [8]:
topk=5
prob, class_number = torch.topk(probs, topk)
for i in range(topk):
    probability = prob[i].item()
    class_name = labels[0][int(class_number[i])]
    print(f"{class_name}: {probability * 100:.2f}%")

Egyptian cat: 37.12%
tabby: 19.35%
tiger cat: 11.50%
Siamese cat: 4.54%
carton: 3.44%


In [9]:
!nvidia-smi
!pip install ipywidgets --trusted-host pypi.org --trusted-host pypi.python.org --trusted-host=files.pythonhosted.org

Tue Aug 12 03:25:11 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   52C    P0             27W /   70W |     298MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [10]:
import numpy as np
import time
import torch.backends.cudnn as cudnn
cudnn.benchmark = True

def rn50_preprocess():
    preprocess = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    return preprocess

# decode the results into ([predicted class, description], probability)
def predict(img_path, model):
    img = Image.open(img_path)
    preprocess = rn50_preprocess()
    input_tensor = preprocess(img)
    input_batch = input_tensor.unsqueeze(0) # create a mini-batch as expected by the model

    # move the input and model to GPU for speed if available
    if torch.cuda.is_available():
        input_batch = input_batch.to('cuda')
        model.to('cuda')

    with torch.no_grad():
        output = model(input_batch)
        # Tensor of shape 1000, with confidence scores over Imagenet's 1000 classes
        sm_output = torch.nn.functional.softmax(output[0], dim=0)

    ind = torch.argmax(sm_output)
    return d[str(ind.item())], sm_output[ind] #([predicted class, description], probability)

def benchmark(model, device="cuda", input_shape=(32, 3, 224, 224), dtype='fp32', nwarmup=50, nruns=100):
    input_data = torch.randn(input_shape)
    input_data = input_data.to(device)
    #if dtype=='fp16':
    #    input_data = input_data.half()

    print("Warm up ...")
    with torch.no_grad():
        for _ in range(nwarmup):
            features = model(input_data)
    torch.cuda.synchronize()
    print("Start timing ...")
    timings = []
    with torch.no_grad():
        for i in range(1, nruns+1):
            start_time = time.time()
            features = model(input_data)
            torch.cuda.synchronize()
            end_time = time.time()
            timings.append(end_time - start_time)
            if i%10==0:
                print('Iteration %d/%d, ave batch time %.2f ms'%(i, nruns, np.mean(timings)*1000))

    print("Input shape:", input_data.size())
    print("Output features size:", features.size())
    print('Average batch time: %.2f ms'%(np.mean(timings)*1000))

In [11]:
benchmark(model, device="cuda")

Warm up ...
Start timing ...
Iteration 10/100, ave batch time 82.21 ms
Iteration 20/100, ave batch time 82.08 ms
Iteration 30/100, ave batch time 82.10 ms
Iteration 40/100, ave batch time 82.19 ms
Iteration 50/100, ave batch time 82.24 ms
Iteration 60/100, ave batch time 82.27 ms
Iteration 70/100, ave batch time 82.29 ms
Iteration 80/100, ave batch time 82.32 ms
Iteration 90/100, ave batch time 82.35 ms
Iteration 100/100, ave batch time 82.37 ms
Input shape: torch.Size([32, 3, 224, 224])
Output features size: torch.Size([32, 1000])
Average batch time: 82.37 ms


In [12]:
traced_model=torch.jit.trace(model, [torch.randn((32,3,224,224)).to('cuda')])

In [14]:
!pip install torch-tensorrt -U --extra-index-url https://pypi.nvidia.com


Looking in indexes: https://pypi.org/simple, https://pypi.nvidia.com
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.7/40.7 kB 13.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 31.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 GB 26.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 125.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 182.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 56.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.1/193.1 MB 100.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 MB 95.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.5/267.5 MB 67.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 288.2/288.2 MB 91.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [13]:
import torch_tensorrt

trt_model=torch_tensorrt.compile(traced_model,
                                 inputs=[torch_tensorrt.Input((32,3,224,224), dtype=torch.float32)],
                                 enabled_precisions={torch.float32}
                                 )

ModuleNotFoundError: No module named 'torch_tensorrt'